## Hướng dẫn làm bài:
1. Sử dụng Pipeline để train mô hình
2. Lưu mô hình sau khi train và các thông tin mô tả vào file model.pkl
3. Nộp file .ipynb lên LMS, không cần nén
4. Copy file model.pkl vào thư mục T:\trained\

In [3]:
# Khai báo thông tin sinh viên
Lop = "1234"
Nhom = "123"
MSSV = "123456"
HoTen = "Nguyen Van A"
SoMay = 0

In [4]:
# Viết code ở đây
# ==========================
# 1. Import libraries
# ==========================
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report
import datetime

# ==========================
# 2. Load dataset
# ==========================
train_file = "Review_MachineLearning\data\dataset_train.csv"

train_df = pd.read_csv(train_file)

# Lấy các cột số
numeric_cols = ['Vmag', 'B-V', 'Plx', 'e_Plx']
for col in numeric_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')

X = train_df[numeric_cols]

# Tạo nhãn Giant/Dwarf
def label_giant_dwarf(sptype):
    if isinstance(sptype, str):
        if 'III' in sptype or 'II' in sptype or 'I' in sptype:
            return 'Giant'
        else:
            return 'Dwarf'
    else:
        return 'Dwarf'

y = train_df['SpType'].apply(label_giant_dwarf)

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split dữ liệu
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ==========================
# 3. Preprocessing
# ==========================
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)],
    voting='soft'
)

# ==========================
# 4. Build pipeline with model
# ==========================
clf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('classifier', voting_clf)
])


# ==========================
# 5. Train and validate
# ==========================
clf.fit(X_train, y_train)

# Validation evaluation
y_val_pred = clf.predict(X_val)
print("Validation set performance:")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

# ==========================
# 6. Save model with metadata
# ==========================
metadata = {
    "Lop": Lop,
    "Nhom": Nhom,
    "MSSV": MSSV,
    "HoTen": HoTen,
    "SoMay": SoMay,
    "TaoLuc": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

save_obj = {"model": clf, "metadata": metadata}
joblib.dump(save_obj, "model.pkl")

print("Model and metadata saved to model.pkl")

Validation set performance:
              precision    recall  f1-score   support

       Dwarf       0.81      0.92      0.86     13201
       Giant       0.65      0.40      0.50      4799

    accuracy                           0.78     18000
   macro avg       0.73      0.66      0.68     18000
weighted avg       0.77      0.78      0.76     18000

Model and metadata saved to model.pkl
